### 🤖 Implement ReAct with LangGraph-What is ReAct?
ReAct (Reasoning + Acting) is a framework where an LLM:

- Reasons step-by-step (e.g. decomposes questions, makes decisions)

- Acts by calling tools like search, calculators, or retrievers

This makes it perfect for Agentic RAG:
✅ Think → Retrieve → Observe → Reflect → Final Answer

In [ ]:
# ============================================================
# Standard Library
# ============================================================

import os
from typing import Annotated, TypedDict, Sequence

# ============================================================
# LangGraph
# ============================================================

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

from langgraph.prebuilt import (
    ToolNode,
    tools_condition,
)
from langchain_core.tools import Tool

from langchain.agents import create_agent
# ============================================================
# LangChain Core
# ============================================================

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
)

# ============================================================
# Tools
# ============================================================

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# ============================================================
# Embeddings
# ============================================================

from langchain_openai import OpenAIEmbeddings

# ============================================================
# Document Loaders
# ============================================================

from langchain_community.document_loaders import WebBaseLoader

# ============================================================
# Text Splitters
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

# ============================================================
# Vector Store
# ============================================================

from langchain_community.vectorstores import FAISS

In [ ]:
# --------------------------
# 1. Create Retriever Tool
# --------------------------

# Load content from blog
docs = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/").load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever()

In [ ]:
retriever.invoke("what are autonomous agents")

In [ ]:
def retriever_tool_func(query: str) -> str:
    print("📚 Using RAGRetriever tool")
    docs = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in docs])

In [ ]:
retriever_tool_func("what are autonomous agents")

In [ ]:
retriever_tool=Tool(
    name="RAGRetriever",
    description="Use this tool to fetch relevant knowledge base info",
    func=retriever_tool_func
)
retriever_tool
print(retriever_tool.name)

In [ ]:
# Wikipedia tool
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
wiki_tool

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
llm=init_chat_model("openai:gpt-4o")

In [ ]:
# ----------------------------
# 2. Define the Agent Node
# ----------------------------



tools = [retriever_tool, wiki_tool]

## create the native Langgraph react agent
react_node = create_agent(
    model=llm,
    tools=tools
)

In [ ]:
# --------------------------
# 3. LangGraph Agent State
# --------------------------

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [ ]:
# --------------------------
# 4. Build LangGraph Graph
# --------------------------

builder = StateGraph(AgentState)

builder.add_node("react_agent", react_node)
builder.set_entry_point("react_agent")
builder.add_edge("react_agent", END)

graph = builder.compile()
graph

In [ ]:
# --------------------------
# 5. Run the ReAct Agent
# --------------------------

if __name__ == "__main__":
    user_query = "What is an agent loop and how does Wikipedia describe autonomous agents?"
    state = {"messages": [HumanMessage(content=user_query)]}
    result = graph.invoke(state)

    print("\n✅ Final Answer:\n", result["messages"][-1].content)

### Tool creation for RAG agents with langgraph 
To create tools for RAG agents using LangGraph, you're essentially building LLM-invocable functions that your agent can call as part of its reasoning + acting loop (ReAct).

LangGraph uses the Tool abstraction from LangChain and fully supports tools for:

- RAG retrieval
- Search
- Wikipedia
- SQL
- Web APIs
- Calculators, etc.

#### ✅ Tool Design Requirements
A LangGraph tool must:

- Have a name
- Have a description (used by the LLM to decide when to use it)
- Have a callable func, accepting a single input (usually str) and returning str

In [ ]:
from langchain.agents import create_agent
# from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage
# from langchain_core.tools import tool
# from langchain_community.utilities import WikipediaAPIWrapper
import arxiv
# ----------------------------
# LLM
# ----------------------------
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

# ----------------------------
# Helper to create retrieval tool
# ----------------------------
def make_retriever_tool_from_text(file_name, tool_name, description):
    docs = TextLoader(file_name, encoding="utf-8").load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
    )

    chunks = splitter.split_documents(docs)

    vectorstore = FAISS.from_documents(
        chunks,
        OpenAIEmbeddings()
    )

    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    def tool_func(query: str) -> str:
        print(f"📚 Using {tool_name}")
        docs = retriever.invoke(query)

        if not docs:
            return "No relevant documents found."

        return "\n\n".join(doc.page_content for doc in docs)

    return Tool(
        name=tool_name,
        description=description,
        func=tool_func,
    )

# ----------------------------
# Wikipedia Tool
# ----------------------------
from langchain_core.tools import tool
from langchain_community.utilities import WikipediaAPIWrapper

wiki = WikipediaAPIWrapper()

@tool
def wikipedia_search(query: str) -> str:
    """Search Wikipedia for general knowledge."""
    print("🌍 Searching Wikipedia...")
    return wiki.run(query)



In [ ]:
# ----------------------------
# Arxiv Tool
# ----------------------------
def arxiv_search(query: str) -> str:
    """Search ArXiv for recent research papers."""

    print("🧪 Searching ArXiv...")

    client = arxiv.Client()

    search = arxiv.Search(
        query=query,
        max_results=2,
        sort_by=arxiv.SortCriterion.Relevance,
    )

    results = []

    for paper in client.results(search):
        results.append(
            f"""
Title: {paper.title}

Summary:
{paper.summary}

URL: {paper.entry_id}
"""
        )

    if not results:
        return "No papers found."

    return "\n\n".join(results)

# ----------------------------
# Internal Tools
# ----------------------------
internal_tool_1 = make_retriever_tool_from_text(
    "sample-docs.txt",
    "InternalTechDocs",
    "Search internal technical documentation."
)

internal_tool_2 = make_retriever_tool_from_text(
    "research-notes.txt",
    "InternalResearchNotes",
    "Search internal research notes."
)



In [ ]:
# ----------------------------
# Agent
# ----------------------------
tools = [
    wikipedia_search,
    arxiv_tool,
    internal_tool_1,
    internal_tool_2,
]

agent = create_agent(
    model=llm,
    tools=tools,
)



In [ ]:
# ----------------------------
# Query
# ----------------------------
query = (
    "What do our internal research notes say about transformer variants, "
    "and what does ArXiv suggest recently?"
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content=query)
        ]
    }
)

print("\n========== FINAL ANSWER ==========\n")
print(response["messages"][-1].content)

In [ ]:
import langchain
import langgraph
import langchain_core
import langchain_openai
import langchain_community

print("langchain:", langchain.__version__)
print("langgraph:", langgraph.__version__)
print("langchain_core:", langchain_core.__version__)
print("langchain_openai:", langchain_openai.__version__)
print("langchain_community:", langchain_community.__version__)

In [ ]:
from importlib.metadata import version

packages = [
    "langchain",
    "langgraph",
    "langchain-core",
    "langchain-openai",
    "langchain-community",
]

for pkg in packages:
    try:
        print(f"{pkg}: {version(pkg)}")
    except Exception as e:
        print(f"{pkg}: {e}")
        

In [ ]:
from typing import get_type_hints

for t in tools:
    print("=" * 80)
    print("Tool:", t.name)

    try:
        func = getattr(t, "func", None)
        print("Function:", func)

        if func:
            print("Annotations:", func.__annotations__)
            print("Resolved:", get_type_hints(func, include_extras=True))

    except Exception as e:
        print("ERROR:", repr(e))

In [6]:
import arxiv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage

# ============================================================
# LLM
# ============================================================

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

# ============================================================
# Internal Retriever Tool Factory
# ============================================================

def make_retriever_tool(file_name: str, tool_name: str, description: str):

    docs = TextLoader(file_name, encoding="utf-8").load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
    )

    chunks = splitter.split_documents(docs)

    vectorstore = FAISS.from_documents(
        chunks,
        OpenAIEmbeddings(),
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 3}
    )

    @tool(tool_name, description=description)
    def retrieve(query: str) -> str:
        """Search internal knowledge."""

        print(f"\n📚 Using {tool_name}")

        docs = retriever.invoke(query)

        if not docs:
            return "No relevant information found."

        return "\n\n".join(d.page_content for d in docs)

    return retrieve


# ============================================================
# Wikipedia Tool
# ============================================================

wiki = WikipediaAPIWrapper()

@tool
def wikipedia_search(query: str) -> str:
    """Search Wikipedia."""

    print("\n🌍 Searching Wikipedia...")

    return wiki.run(query)


# ============================================================
# Arxiv Tool (Latest API)
# ============================================================

@tool
def arxiv_search(query: str) -> str:
    """Search recent papers from ArXiv."""

    print("\n🧪 Searching ArXiv...")

    client = arxiv.Client()

    search = arxiv.Search(
        query=query,
        max_results=2,
        sort_by=arxiv.SortCriterion.Relevance,
    )

    papers = []

    for paper in client.results(search):
        papers.append(
            f"""
Title: {paper.title}

Published: {paper.published.date()}

Summary:
{paper.summary[:1200]}

URL:
{paper.entry_id}
"""
        )

    return "\n\n".join(papers) if papers else "No papers found."


# ============================================================
# Internal RAG Tools
# ============================================================

internal_docs = make_retriever_tool(
    "sample-docs.txt",
    "internal_docs_search",
    "Search internal technical documentation."
)

research_notes = make_retriever_tool(
    "research-notes.txt",
    "research_notes_search",
    "Search internal research notes."
)

# ============================================================
# Agent
# ============================================================

agent = create_agent(
    model=llm,
    tools=[
        wikipedia_search,
        arxiv_search,
        internal_docs,
        research_notes,
    ],
    system_prompt="""
You are an AI research assistant.

Use:
- research_notes_search for research notes
- internal_docs_search for company documents
- arxiv_search for latest papers
- wikipedia_search for general knowledge

Always use tools whenever external knowledge is required.
""",
)

# ============================================================
# Query
# ============================================================

query = """
What do our internal research notes say about transformer variants,
and what does ArXiv suggest recently?
"""

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content=query)
        ]
    }
)

print("\n" + "=" * 80)
print(response["messages"][-1].content)
print("=" * 80)

RuntimeError: function 'conv1d' already has a docstring

In [7]:
import torch
print(torch.__version__)

RuntimeError: function 'conv1d' already has a docstring

In [8]:
import importlib.metadata as md

packages = [
    "torch",
    "transformers",
    "sentence-transformers",
    "langchain",
    "langgraph",
    "langchain-core",
    "langchain-community",
]

for p in packages:
    try:
        print(f"{p}: {md.version(p)}")
    except Exception:
        print(f"{p}: NOT INSTALLED")

torch: 2.13.0
transformers: 5.14.1
sentence-transformers: 5.6.1
langchain: 1.3.14
langgraph: 1.2.9
langchain-core: 1.4.9
langchain-community: 0.4.2


In [11]:
!pip show torch
!pip show transformers
!pip show sentence-transformers

Name: torch
Version: 2.13.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License-Expression: Apache-2.0 AND Apache-2.0 WITH LLVM-exception AND BSD-2-Clause AND BSD-3-Clause AND BSL-1.0 AND MIT
Location: c:\Users\SHAILENDRA\anaconda3\envs\env_langchain\Lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: sentence-transformers, torchvision
Name: transformers
Version: 5.14.1
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License

In [ ]:
!pip uninstall -y sentence-transformers transformers torch torchvision torchaudio

!pip install torch
!pip install transformers
!pip install sentence-transformers